# Abstract

# Dataset Description

## Import required modules and download csv file.

In [637]:
import kagglehub
import polars as pl
import os
path = kagglehub.dataset_download("divyansh22/online-gaming-anxiety-data")

print("Path to dataset files:", path)
csv = os.path.join(path, "GamingStudy_data.csv")
df = pl.read_csv(csv, encoding="latin1", null_values="NA")

Path to dataset files: C:\Users\jlgfd\.cache\kagglehub\datasets\divyansh22\online-gaming-anxiety-data\versions\3


# Data Cleaning and Preprocessing

## Drop Unnecessary Columns
These columns were dropped as this study will utilize the aggregate variables rather than individual questionnaire items:
- `S. No.` & `Timestamp` (not useful for statistical modeling)
- `GAD1` to `GAD7` (redundant, represented by the total score `GAD_T`)
- `SWL1` to `SWL5` (redundant, represented by `SWL_T`)
- `SPIN1` to `SPIN17` (redundant, represented by `SPIN_T`)
- `highestleague` (100% null/missing values)
- `Residence_ISO3` & `Birthplace_ISO3` (redundant, represented by `Residence` and `Birthplace` respectively)
- `accept` (participant consent column, contains no predictive information for modeling)
- `reference` (contains no predictive information for modeling)

In [638]:
dropped_columns = [
    "S. No.", "Timestamp", "GAD1", "GAD2", "GAD3", "GAD4", "GAD5", "GAD6", "GAD7", 
    "SWL1", "SWL2", "SWL3", "SWL4", "SWL5", "SPIN1", "SPIN2", "SPIN3", "SPIN4", 
    "SPIN5", "SPIN6", "SPIN7", "SPIN8", "SPIN9", "SPIN10", "SPIN11", "SPIN12", 
    "SPIN13", "SPIN14", "SPIN15", "SPIN16", "SPIN17", "highestleague",
    "Residence_ISO3", "Birthplace_ISO3", "accept", "Reference"
]

df_clean = df.drop(dropped_columns)
df_clean.head()

GADE,Game,Platform,Hours,earnings,whyplay,League,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Playstyle,GAD_T,SWL_T,SPIN_T
str,str,str,i64,str,str,str,i64,i64,str,i64,str,str,str,str,str,i64,i64,i64
"""Not difficult at all""","""Skyrim""","""Console (PS, Xbox, ...)""",15,"""I play for fun""","""having fun""","""N/A""",0,1,"""Male""",25,"""Unemployed / between jobs""","""Bachelor (or equivalent)""","""USA""","""USA""","""Singleplayer""",1,23,5
"""Somewhat difficult""","""Other""","""PC""",8,"""I play for fun""","""having fun""","""N/A""",2,1,"""Male""",41,"""Unemployed / between jobs""","""Bachelor (or equivalent)""","""USA""","""USA""","""Multiplayer - online - with st…",8,16,33
"""Not difficult at all""","""Other""","""PC""",0,"""I play for fun""","""having fun""","""n/a""",0,4,"""Female""",32,"""Employed""","""Bachelor (or equivalent)""","""Germany""","""Germany""","""Singleplayer""",8,17,31
"""Not difficult at all""","""Other""","""PC""",20,"""I play for fun""","""improving""","""N/A""",5,2,"""Male""",28,"""Employed""","""Bachelor (or equivalent)""","""USA""","""USA""","""Multiplayer - online - with on…",0,17,11
"""Very difficult""","""Other""","""Console (PS, Xbox, ...)""",20,"""I play for fun""","""having fun""","""n/a""",1,1,"""Male""",19,"""Employed""","""High school diploma (or equiva…","""USA""","""South Korea""","""Multiplayer - online - with st…",14,14,13


## Outlier Treatment

Drop unrealistic values in the `Hours` column (participants reporting >80 hours of gaming per week).

In [639]:
df_clean["Hours"].describe()

statistic,value
str,f64
"""count""",13434.0
"""null_count""",30.0
"""mean""",22.247357
"""std""",70.284502
"""min""",0.0
"""25%""",12.0
"""50%""",20.0
"""75%""",28.0
"""max""",8000.0


In [640]:
df_clean = df_clean.filter(pl.col("Hours") <= 80)
df_clean.shape

(13382, 19)

# Null Cleaning
Each column was checked for null values. Whether to drop or impute nulls was decided based on the proportion and nature of the missingness.

In [641]:
df_clean.null_count()

GADE,Game,Platform,Hours,earnings,whyplay,League,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Playstyle,GAD_T,SWL_T,SPIN_T
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
645,0,0,0,0,0,1739,89,23,0,0,38,0,0,0,0,0,0,646


Drop instances that has null values for streams, Narcissism, Work as they are small enough(less than 1% of the population) that they won't skew data

In [642]:
df_clean = df_clean.drop_nulls("Narcissism")

In [643]:
df_clean = df_clean.drop_nulls("streams")

In [644]:
df_clean = df_clean.drop_nulls("Work")

*Take note the "Work" column includes unemployed, students and employment so there's no valid reason to have it as null*

## Standardizing the League and whyplay Columns

Since the `League` and `whyplay` columns are free-text response fields, they contain highly unstandardized entries (custom comments, typos, and mixtures). We standardize them into discrete categories.

#### 1. League Rank Standardization:
- **Master+**: Challenger, Master, Grandmaster, GM, and common typos (*challenjour*, etc.).
- **Diamond**: Diamond, Dia, and abbreviations (*d1* to *d5*).
- **Platinum**: Platinum, Plat, and spelling variations (*plarinum*, *plantinum*).
- **Gold**: Gold and typos (*glod*, *golld*, *goled*).
- **Silver**: Silver and spelling variations (*sliver*, *siver*, *silber*).
- **Bronze**: Bronze and spelling variations (*bronce*, *broze*).
- **Unranked**: Captures all non-league values, including blank/unranked responses, CS:GO ranks, and noise.

#### 2. whyplay (Reason for Playing) Standardization:
- **Having Fun**: Joy, pleasure, or fun-related motivations.
- **Improving**: Skill progression, learning, and getting better.
- **Winning**: Competition, victory, and rank-climbing.
- **Relaxing**: De-stressing, unwinding, and casual play.
- **Mixed / All**: Custom responses specifying multiple reasons.
- **Escapism**: Passing time, distraction, and escaping reality.
- **Other**: Custom answers.

In [645]:
df_clean = df_clean.with_columns(pl.col("whyplay").str.to_lowercase())
df_clean = df_clean.with_columns(pl.col("League").str.to_lowercase())

In [646]:
# Apply simplified Polars mapping expressions
league_lower = pl.col("League").fill_null("unranked").str.to_lowercase().str.strip_chars()

# Clean CS:GO-specific rank terms from the column series so they don't trigger standard rank detection
cleaned_league = league_lower
csgo_patterns = [
    r"\bgold\s*nova\s*(?:master|[1-4]|i{1,3}|iv)?\b",
    r"\bsilver\s*elite\s*(?:master)?\b",
    r"\bsilver\s*master\s*elite\b",
    r"\bmaster\s*guardian\s*(?:elite|[1-2]|ii|i)?\b",
    r"\bdistinguished\s*master\s*guardian\b",
    r"\blegendary\s*eagle\s*(?:master)?\b",
    r"\bsupreme\s*master\s*(?:first\s*class)?\b",
    r"\bglobal\s*elite\b",
    r"\bmg[12]\b",
    r"\bmge\b",
    r"\ble[m]?\b",
    r"\bdmg\b",
]
for pattern in csgo_patterns:
    cleaned_league = cleaned_league.str.replace_all(pattern, "")

cleaned_league = cleaned_league.str.strip_chars()

# CS:GO keywords check
is_csgo_keyword = league_lower.str.contains("cs:?go|counter\s*strike|global\s*offensive")
is_other_game = league_lower.str.contains("lol|league|sc2|starcraft")

# Check if cleaned_league contains any standard ranks
has_any_standard_rank = (
    cleaned_league.str.contains(r"\b(challenger|master|grandmaster|grand\s+master|gm|challeneger|challenjour|charrenjour)s?\d*\b") |
    cleaned_league.str.contains(r"\b(diamond|dia|dim|diamomd)s?\d*\b|\b(d\s*[1-5])\b") |
    cleaned_league.str.contains(r"\b(platinum|platinium|platinuim|platin|platine|platnium|plat|plarinum|plantinum|pplatinum|platnum|platinumm|platinun|platium)s?\d*\b|\b(p\s*[1-5])\b") |
    cleaned_league.str.contains(r"\b(gold|glod|golld|goled|golden|golf)s?\d*\b|\b(g\s*[1-5])\b") |
    cleaned_league.str.contains(r"\b(silver|sliver|siver|silber)s?\d*\b|\b(s\s*[1-5])\b") |
    cleaned_league.str.contains(r"\b(bronze|bronce|broze)s?\d*\b|\b(b\s*[1-5])\b")
)

is_csgo_only = is_csgo_keyword & ~is_other_game & ~has_any_standard_rank

is_currently_unranked = (
    cleaned_league.str.contains(r"\b(currently|now|atm|season\s*5|this\s*season|at\s*the\s*moment)\s*(unranked|unrank|unplaced|placements|placement|provisional)\b") |
    (cleaned_league.str.contains("unranked|unrank|unplaced") & cleaned_league.str.contains("previously|was|last\s*season|s4|season\s*4"))
)

deserve_challenger = cleaned_league.str.contains(r"\b(deserve|deserves|should\s*be)\s*(challenger|challenjour|master)\b")

df_clean = df_clean.with_columns(
    pl.when(is_csgo_only | is_currently_unranked | (cleaned_league == "") | (cleaned_league == ",") | (cleaned_league == "/") | (cleaned_league == "-") | (cleaned_league == "()"))
    .then(pl.lit("Unranked"))
    .when(cleaned_league.str.contains(r"\b(challenger|master|grandmaster|grand\s+master|gm|challeneger|challenjour|charrenjour)s?\d*\b") & ~deserve_challenger)
    .then(pl.lit("Master+"))
    .when(cleaned_league.str.contains(r"\b(diamond|dia|dim|diamomd)s?\d*\b|\b(d\s*[1-5])\b") | cleaned_league.str.starts_with("dia"))
    .then(pl.lit("Diamond"))
    .when(cleaned_league.str.contains(r"\b(platinum|platinium|platinuim|platin|platine|platnium|plat|plarinum|plantinum|pplatinum|platnum|platinumm|platinun|platium)s?\d*\b|\b(p\s*[1-5])\b") | cleaned_league.str.starts_with("plat") | cleaned_league.str.starts_with("platin") | cleaned_league.str.starts_with("plari"))
    .then(pl.lit("Platinum"))
    .when(cleaned_league.str.contains(r"\b(gold|glod|golld|goled|golden|golf)s?\d*\b|\b(g\s*[1-5])\b") | cleaned_league.str.starts_with("gold"))
    .then(pl.lit("Gold"))
    .when(cleaned_league.str.contains(r"\b(silver|sliver|siver|silber)s?\d*\b|\b(s\s*[1-5])\b") | cleaned_league.str.starts_with("silver") | cleaned_league.str.starts_with("sliver"))
    .then(pl.lit("Silver"))
    .when(cleaned_league.str.contains(r"\b(bronze|bronce|broze)s?\d*\b|\b(b\s*[1-5])\b") | cleaned_league.str.starts_with("bronze") | cleaned_league.str.starts_with("bronce"))
    .then(pl.lit("Bronze"))
    .otherwise(pl.lit("Unranked"))
    .alias("League_Standardized")
)

wp_lower = pl.col("whyplay").fill_null("other").str.to_lowercase().str.strip_chars()

has_fun = wp_lower.str.contains("fun|enjoy|pleasure|joy|happi|entertain|amuse|laugh|smile")
has_improv = wp_lower.str.contains("improv|learn|better|skill|practic|grow|perform|master|train|get\s+good")
has_win = wp_lower.str.contains("win|compet|success|rank|climb|goal|achiev|destroy|crush|kda|defeat|victor")
has_relax = wp_lower.str.contains("relax|stress|calm|unwind|chill|casual|peace|sooth")

is_social_time = wp_lower.str.contains("friend|special|people|family|partner|girlfriend|boyfriend|wife|husband|someone|others|son|daughter|kids|play\s+with")
has_time_sink = wp_lower.str.contains("time") & ~is_social_time & wp_lower.str.contains("pass|kill|wast|sink|occup|spend")

has_escapism = wp_lower.str.contains("escap|distract|bored|reality|real\s+life|real\s+world|forget|hide|trouble|nervous|worr|occup") | has_time_sink

has_mixed_words = wp_lower.str.contains(r"\b(all|both|everything|mix|mixture|combination|combo)\b")
has_option_letters = wp_lower.str.contains(r"\b[a-d]\b.*\b[a-d]\b")

category_count = (
    has_fun.cast(pl.Int32) +
    has_improv.cast(pl.Int32) +
    has_win.cast(pl.Int32) +
    has_relax.cast(pl.Int32) +
    has_escapism.cast(pl.Int32)
)

is_mixed = has_mixed_words | has_option_letters | (category_count > 1)

df_clean = df_clean.with_columns(
    pl.when(is_mixed)
    .then(pl.lit("Mixed / All"))
    .when(has_fun)
    .then(pl.lit("Having Fun"))
    .when(has_improv)
    .then(pl.lit("Improving"))
    .when(has_win)
    .then(pl.lit("Winning"))
    .when(has_relax)
    .then(pl.lit("Relaxing"))
    .when(has_escapism)
    .then(pl.lit("Escapism"))
    .otherwise(pl.lit("Other"))
    .alias("whyplay_Standardized")
)

print(df_clean.select(["League", "League_Standardized"]).head(5))
print(df_clean.select(["whyplay", "whyplay_Standardized"]).head(5))


<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:43: SyntaxWarning: invalid escape sequence '\s'
<>:70: SyntaxWarning: invalid escape sequence '\s'
<>:74: SyntaxWarning: invalid escape sequence '\s'
<>:77: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:43: SyntaxWarning: invalid escape sequence '\s'
<>:70: SyntaxWarning: invalid escape sequence '\s'
<>:74: SyntaxWarning: invalid escape sequence '\s'
<>:77: SyntaxWarning: invalid escape sequence '\s'
C:\Users\jlgfd\AppData\Local\Temp\ipykernel_2704\2639331753.py:26: SyntaxWarning: invalid escape sequence '\s'
  is_csgo_keyword = league_lower.str.contains("cs:?go|counter\s*strike|global\s*offensive")
C:\Users\jlgfd\AppData\Local\Temp\ipykernel_2704\2639331753.py:43: SyntaxWarning: invalid escape sequence '\s'
  (cleaned_league.str.contains("unranked|unrank|unplaced") & cleaned_league.str.contains("previously|was|last\s*season|s4|season\s*4"))
C:\Users\jlgfd\AppData\Local\Temp\ipyker

shape: (5, 2)
┌────────┬─────────────────────┐
│ League ┆ League_Standardized │
│ ---    ┆ ---                 │
│ str    ┆ str                 │
╞════════╪═════════════════════╡
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
└────────┴─────────────────────┘
shape: (5, 2)
┌────────────┬──────────────────────┐
│ whyplay    ┆ whyplay_Standardized │
│ ---        ┆ ---                  │
│ str        ┆ str                  │
╞════════════╪══════════════════════╡
│ having fun ┆ Having Fun           │
│ having fun ┆ Having Fun           │
│ having fun ┆ Having Fun           │
│ improving  ┆ Improving            │
│ having fun ┆ Having Fun           │
└────────────┴──────────────────────┘


In [647]:
# Save original vs standardized column mappings to an editable text file
with pl.Config(tbl_rows=-1, fmt_str_lengths=1000, tbl_width_chars=10000):
    league_mapping = str(df_clean.group_by(["League", "League_Standardized"]).len().sort("len", descending=True))
    whyplay_mapping = str(df_clean.group_by(["whyplay", "whyplay_Standardized"]).len().sort("len", descending=True))

# Drop the original columns as they won't be needed anymore    
df_clean.drop_in_place("League")
df_clean.drop_in_place("whyplay")

with open("standardization_output.txt", "w", encoding="utf-8") as f:
    f.write("=== ORIGINAL LEAGUE vs STANDARDIZED LEAGUE ===\n")
    f.write(league_mapping)
    f.write("\n\n=== ORIGINAL WHYPLAY vs STANDARDIZED WHYPLAY ===\n")
    f.write(whyplay_mapping)

# Save aggregate (overall distribution) of standardized columns to a separate file
with pl.Config(tbl_rows=-1, fmt_str_lengths=1000, tbl_width_chars=10000):
    league_aggregate = str(df_clean["League_Standardized"].value_counts(sort=True))
    whyplay_aggregate = str(df_clean["whyplay_Standardized"].value_counts(sort=True))

with open("standardization_aggregate.txt", "w", encoding="utf-8") as f:
    f.write("=== AGGREGATE STANDARDIZED LEAGUE COUNTS ===\n")
    f.write(league_aggregate)
    f.write("\n\n=== AGGREGATE STANDARDIZED WHYPLAY COUNTS ===\n")
    f.write(whyplay_aggregate)

In [648]:
df_clean.null_count()

GADE,Game,Platform,Hours,earnings,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Playstyle,GAD_T,SWL_T,SPIN_T,League_Standardized,whyplay_Standardized
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
640,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,626,0,0


# References
- <https://www.kaggle.com/datasets/divyansh22/online-gaming-anxiety-data/data>
- <https://osf.io/preprints/psyarxiv/mfajz_v1>